In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from app.core.config import settings
from dotenv import load_dotenv

load_dotenv()

# 创建连接池
checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()

In [2]:
# 初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
# 创建Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
)

In [3]:
config = {"configurable": {"thread_id": "day06-checkpointer-test"}}
# 第一次，告知自己的信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫小明，今年30岁。")]},
    config=config
)
# 第二次，询问信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我今年多大？")]},
    config=config
)
print(result["messages"][-1].content)

你今年 **30岁** 啦！😊

你刚才自己说的：“我叫小明，今年30岁。”
